# 01 - Ingestão das Fontes

## Objetivo

Realizar a leitura dos arquivos disponibilizados para o case, identificando suas estruturas, formatos e principais características.

## Atividades
- Funções
- Leitura dos arquivos de origem
- Validação da estrutura (schema)
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade

Observação os arquivos de origem foram mantidos em /Workspace/case_data_engineer/sources

#### Funções

In [0]:
%pip install openpyxl
%restart_python

In [0]:
from datetime import datetime, timedelta
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from dateutil.relativedelta import *
from pyspark.storagelevel import StorageLevel
from delta.tables import *
from pytz import timezone
from array import *
import pandas as pd

#df_legado_regioes_pipe

#### Leitura do arquivo de origem

In [0]:
df_legado_regioes_pipe = (spark.read
    .option("header", "true")
    .option("delimiter", "|")
    .option("inferSchema", "true")
    .csv("/Workspace/case_data_engineer/sources/legado_regioes_pipe.txt")
)

#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade

In [0]:
display(df_legado_regioes_pipe)

df_legado_regioes_pipe.printSchema()

# Quantidade de linhas e colunas
print(f"Linhas: {df_legado_regioes_pipe.count()}")
print(f"Colunas: {len(df_legado_regioes_pipe.columns)}")

# nulos

from pyspark.sql.functions import col, count, when

nulos = df_legado_regioes_pipe.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_legado_regioes_pipe.columns
])

nulos.show()

#estatistica descritiva
df_legado_regioes_pipe.describe().show()

### Principais observações

- Excluir linha sem regional_code, se as outras colunas também não tiver registro válido.
- Renomear nome das colunas.
- Padronizar nomes das variáveis categóricas.
- Manter somente siglas na coluna state.
- Manter a coluna active_flag (caso ocorra nova carga com registro booleano)

#df_erp_pedidos_itens_2025


#### Leitura do arquivo de origem

In [0]:
df_pedidos_itens2025 = (
    spark.read
    .option("header", "true")
    .option("delimiter", ",")
    .option("inferSchema", "true")
    .csv("/Workspace/case_data_engineer/sources/erp_pedidos_itens_2025.csv")
)
display(df_pedidos_itens2025)

#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade

In [0]:
df_pedidos_itens2025.printSchema()

# Quantidade de linhas e colunas
print(f"Linhas: {df_pedidos_itens2025.count()}")
print(f"Colunas: {len(df_pedidos_itens2025.columns)}")

# nulos
from pyspark.sql.functions import col, count, when

nulos = df_pedidos_itens2025.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pedidos_itens2025.columns
])
nulos.show()

#estatistica descritiva
df_pedidos_itens2025.describe().show()


In [0]:
#verificando valores distintos

from pyspark.sql.functions import countDistinct

df_pedidos_itens2025.select([
    countDistinct(col(c)).alias(c)
    for c in df_pedidos_itens2025.columns
]).show()

### Principais observações

- Campo null na coluna quantily deve ser aplicado o seguinte racional, por regra de negócio: quantily = total_item / unit_price.
- Tratar valores negativos na coluna quantily como positivos (por regra de negócio)
- Tratar valores negativos na coluna total_item como positivos (por regra de negócio)
- Tratar "," por "." na coluna unit_price
- Alterar schema coluna unit_prece paa double
- Renomear nome das colunas.
- Padronizar nome na variável categorica item_status e alterar para caixa alta.
- Por regra de negócio campo null na variável categórica deve ser classificado como "sem_status".
- Padronizar colunas order_id e product_code.


#df_pedidos_cabecalho2025


#### Leitura do arquivo de origem

In [0]:
df_pedidos_cabecalho2025 = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "true")
    .csv("/Workspace/case_data_engineer/sources/erp_pedidos_cabecalho_2025.csv")
)
display(df_pedidos_cabecalho2025)


#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade

In [0]:
df_pedidos_cabecalho2025.printSchema()


# Quantidade de linhas e colunas
print(f"Linhas: {df_pedidos_cabecalho2025.count()}")
print(f"Colunas: {len(df_pedidos_cabecalho2025.columns)}")

# nulos
from pyspark.sql.functions import col, count, when

nulos = df_pedidos_cabecalho2025.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pedidos_cabecalho2025.columns
])
nulos.show()


#estatistica descritiva
df_pedidos_cabecalho2025.describe().show()


### Principais observações

- Campo null na coluna staus_order deve ser classificado como "sem_status", por regra de negócio.
- Padronizar colunas customer_code e order_id.
- Identificado 3 registros com linhas duplicadas na coluna order_id. VALIDAR COM NEGÓCIO A LÓGICA DE ELIMINAÇÃO DE UMA DAS LINHAS
- Padronizar a coluna status_order.
- Campo null na coluna status_order deve ser classificado como "sem status"(por regra de negócio).
- Padronização das fontes caixa alta e erro ortográfico na coluna status_order.
- Criar coluna canal e prioridade com os registros da coluna payment_details.
- Ajustar os nomes dos registros das colunas canal e prioridade para caixa alta, português e null como "sem status".
- Por regra de negócio registro N/A na coluna gross_amount deve ser desconsiderado.
- Ajustar "," por "." na coluna gross_amount.
- Por regra de negócio, valores negativos devem ser corrigido para valores positivos na coluna net_amount.
- Alterar schema das colunas order_date e promised_date de string para date.
- Renomear nomes das colunas.

#df_vendedores

#### Leitura do arquivo de origem

In [0]:
df_vendedores = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "true")
    .csv("/Workspace/case_data_engineer/sources/vendedores.csv")

)
display(df_vendedores)

#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade


In [0]:
df_vendedores.printSchema()

# Quantidade de linhas e colunas
print(f"Linhas: {df_vendedores.count()}")
print(f"Colunas: {len(df_vendedores.columns)}")

# nulos
from pyspark.sql.functions import col, count, when

nulos = df_vendedores.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_vendedores.columns
])
nulos.show()

#estatistica descritiva
df_vendedores.describe().show()


### Principais observações

- Campo null nas colunas canal_id, regional_code e status.
- Padronizar coluna canal_id.
- Padronizar coluna regional_code.
- Alterar schema da coluna hire_date para date.
- Padronizar coluna status e tratar campo null como "sem status".
- Por regar de negócio, desconsiderar vendedor 04 duplicado, se canal_id = CH99
- Desconsiderar vendedor 08 classificado como vendedor duplicado na coluna seller_name.
- Tratar null na coluna canal_id como CH99.

#df_comercial_canais

#### Leitura do arquivo de origem

In [0]:
df_comercial_canais = pd.read_excel(
    "/Workspace/case_data_engineer/sources/comercial_canais.xlsx",
    engine="openpyxl"
)
display(df_comercial_canais)

#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade

In [0]:
df_comercial_canais.info()



### Principais observações

- Alterar schema de todas as colunas.
- Padronizar coluna id_canal.
- Padronizar colunas nome_canal, ativo, observação e tipo_canal mantendo todas os registros em caixa alta.
- Coluna nome_canal com registro null classificar como "sem nome_canal". 
- Coluna ativo com registro null classificar como "sem info". 
- Coluna observação com registro null classificar como "Não declarado". 

#df_crm_clientes_export

#### Leitura do arquivo de origem

In [0]:
df_crm_clientes_export = pd.read_excel(
    "/Workspace/case_data_engineer/sources/crm_clientes_export.xlsx",
    engine="openpyxl"
)
display(df_crm_clientes_export)


#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade

In [0]:
df_crm_clientes_export.info()



### Principais observações

- Alterar schema de todas as colunas.
- Renomear nomes das colunas.
- Padronizar os registros de todas as colunas categóricas. Manter todos os registros com caixa alta.
- Tratar registros null como " sem + (nome da coluna)".
- Coluna data_cadastro padronizar e tratar como date.


# df_logistica_entregas

#### Leitura do arquivo de origem


In [0]:
df_logistica_entregas = (
    spark.read
    .option("multiline", "true")
    .json("/Workspace/case_data_engineer/sources/logistica_entregas.json")
)
display(df_logistica_entregas)

#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade


In [0]:
df_logistica_entregas.printSchema()

# Quantidade de linhas e colunas
print(f"Linhas: {df_logistica_entregas.count()}")
print(f"Colunas: {len(df_logistica_entregas.columns)}")

#estatistica descritiva
df_logistica_entregas.describe().show()



### Principais observações

- Renomear nomes das colunas.
- Transforma estruturas aninhadas em colunas simples, nas colunas carrier, destination e timestamps.
- Alterar schema
- Padronizar registros
- Tratar registros null como " sem + (nome da coluna)".
- Remover duplicatas, caso exista mais de um registro para o mesmo delivery_id.


#df_cadastro_produtos

#### Leitura do arquivo de origem


In [0]:
df_cadastro_produtos = (
    spark.read
    .option("multiline", "true")  
    .json("/Workspace/case_data_engineer/sources/cadastro_produtos_api_dump.json")
)

display(df_cadastro_produtos)

#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade


In [0]:

df_cadastro_produtos.printSchema()

# Quantidade de linhas e colunas
print(f"Linhas: {df_cadastro_produtos.count()}")
print(f"Colunas: {len(df_cadastro_produtos.columns)}")

#estatistica descritiva
df_cadastro_produtos.describe().show()



### Principais observações

- Renomear nomes das colunas.
- Transforma estruturas aninhadas em colunas simples.
- Alterar schema
- Padronizar registros
- Tratar registros null como " sem + (nome da coluna)".


#df_atendimento_ocorrencias


#### Leitura do arquivo de origem

In [0]:
df_atendimento_ocorrencias = (
    spark.read
    .json("/Workspace/case_data_engineer/sources/atendimento_ocorrencias.ndjson")
)
display(df_atendimento_ocorrencias)

#### Validação da estrutura
- Schema
- Contagem de registros
- Análise exploratória inicial
- Identificação de possíveis problemas de qualidade


In [0]:
df_atendimento_ocorrencias.printSchema()

# Quantidade de linhas e colunas
print(f"Linhas: {df_atendimento_ocorrencias.count()}")
print(f"Colunas: {len(df_atendimento_ocorrencias.columns)}")

#estatistica descritiva
df_atendimento_ocorrencias.describe().show()




### Principais observações

- Renomear nomes das colunas.
- Transforma estruturas aninhadas em colunas simples, coluna metadata.
- Alterar schema
- Padronizar registros
- Tratar registros null como " sem + (nome da coluna)".
- Tratar coluna created_at com data e padronizar.